# 02b - Engineer Features and Targets

## Purpose
Convert raw GEE observations into per-sample model features and future/t+1 targets using fast local Pandas operations.

## Inputs
- `data/processed/sample_index.csv`
- `data/raw/<label>/<sample_id>/gee_observations.csv`

## Outputs
- `data/processed/<label>/<sample_id>/gee_features.csv`
- `data/processed/<label>/<sample_id>/gee_targets.csv`

## Notes
This notebook applies leakage-controlled imputation before deriving lags, rolling statistics, trend features, seasonality, urban-pressure features, SAR fallback features, and t+1 targets. It does not call Earth Engine and does not generate combined `gee_features_all.csv` or `gee_targets_all.csv` files.

## 1. Configure Local Engineering Paths
Defines the project root, data folder, sample index path, and optional sample limit for local feature engineering.

In [ ]:
from pathlib import Path
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_INDEX_PATH = DATA_DIR / "processed" / "sample_index.csv"
SAMPLE_LIMIT = None

## 2. Load Feature Engineering Helpers
Reloads the GEE feature module and binds the local engineering, loading, and leakage-check helpers.

In [ ]:
import src.features.gee_features as gee_feature_module
importlib.reload(gee_feature_module)

FeatureEngineeringConfig = gee_feature_module.FeatureEngineeringConfig
engineer_all_samples = gee_feature_module.engineer_all_samples
load_per_sample_features = gee_feature_module.load_per_sample_features
load_per_sample_targets = gee_feature_module.load_per_sample_targets
assert_no_target_leakage = gee_feature_module.assert_no_target_leakage

## 3. Generate Per-Sample Features and Targets
Reads raw observations from `data/raw`, applies the configured imputation strategy, and writes per-sample `gee_features.csv` and `gee_targets.csv`.

In [ ]:
config = FeatureEngineeringConfig(
    data_dir=str(DATA_DIR),
    sample_index_csv=str(SAMPLE_INDEX_PATH),
    force=True,
    verbose=True,
    ffill_enabled=True,
    leading_bfill_limit=2,
)

summary = engineer_all_samples(config, sample_limit=SAMPLE_LIMIT)
summary

## 4. Validate Engineered Outputs
Loads all per-sample outputs, checks that feature files do not contain future target columns, and previews the feature table.

In [ ]:
features = load_per_sample_features(SAMPLE_INDEX_PATH)
targets = load_per_sample_targets(SAMPLE_INDEX_PATH)
assert_no_target_leakage(features)
assert not any(column.startswith("target_") for column in features.columns)
print("Feature rows:", len(features))
print("Target rows:", len(targets))
features.head()